<a href="https://colab.research.google.com/github/duckmaster168/Research_MPGELU_Testing_Students/blob/main/Test_2(with%20both%20MLB%20and%20CNN%20with%20F1Score%2C%20Lambda%20Drift%2C%20Gradient%20Norms).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Environment Setup & Core Functions**

In [15]:
import os
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import f1_score
from tqdm import tqdm

# Local project modules
from DataTransforms import DataTransforms
from MyCNN import MyCNN
from MyMLP import MyMLP
from Trainer import Trainer
from activations import get_activation

# Select hardware accelerator (defaults safely to CPU if none available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing experiments on device: {device}")

# Classification accuracy metric
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    return (correct / len(y_pred)) * 100.0

# Directory for persistent records
checkpoint_dir = "experiment_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

Executing experiments on device: cpu


# **Data Pipeline Initialization**


In [16]:
# Initialize data pipelines with normalization and augmentation
transforms_handler = DataTransforms(dataset="cifar10", use_augmentation=True, use_stats=True)

train_dataset = datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transforms_handler.get_train_transform()
)
test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transforms_handler.get_test_transform()
)

# Dynamically set pin_memory based on hardware availability to avoid warnings
use_cuda = torch.cuda.is_available()

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=use_cuda)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=use_cuda)

print(f"Dataset ready | Training samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")

Dataset ready | Training samples: 50000 | Test samples: 10000


# **Phase 1(CNN):**

# **Execution Loop — CNN Baselines Training & Auto-Saving**

In [ ]:
print("=======================================================================")
print("                   PHASE 1: TRAINING CNN BASELINES                    ")
print("=======================================================================")

cnn_arch = [32, 32, 32, "MaxPool", 64, 64, "MaxPool", 128, 128, "MaxPool"]
baselines = ["relu", "leakyrelu", "gelu", "pgelu", "lambdagelu", "mpgelu"]

epochs = 60
cnn_results = {}

for name in baselines:
    print("=======================================================================")
    print(f"               TRAINING WITH ACTIVATION: {name.upper()}               ")
    print("=======================================================================")

    act_class = get_activation(name)

    model = MyCNN(input_shape=3, output_shape=10, activation=act_class, params=cnn_arch).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    trainer = Trainer(
        model=model,
        loss_fn=loss_fn,
        optimizer=optimizer,
        calculate_accuracy=accuracy_fn,
        device=device,
        loss_steps=1
    )

    history = {
        "train_loss": [], "train_acc": [],
        "test_loss": [], "test_acc": [], "test_f1": [],
        "grad_norms": [], "lambdas": []
    }

    epoch_bar = tqdm(range(1, epochs + 1), desc=f"Training {name.upper()}", unit="epoch")

    for epoch in epoch_bar:
        tr_loss, tr_acc, grads, lams = trainer.train(train_loader, epoch=epoch)

        # Evaluate test metrics and F1-score every 5 epochs (or on the final epoch)
        if epoch % 5 == 0 or epoch == epochs:
            te_loss, te_acc = trainer.test(test_loader, epoch=epoch)

            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for X, y in test_loader:
                    X, y = X.to(device), y.to(device)
                    preds = model(X).argmax(dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(y.cpu().numpy())

            epoch_f1 = f1_score(all_targets, all_preds, average="macro") * 100.0

            # Calculate mean gradient norm and lambda for clean display
            mean_grad = sum(grads.values()) / len(grads) if grads else 0.0
            mean_lam = sum(lams) / len(lams) if lams else 0.0

            print(f"Epoch: {epoch}")
            print("-----------------------------------------------------------------------")
            print(f"Training Loss: {tr_loss:.5f} | Training Accuracy: {tr_acc:.5f}%")
            print(f"Test Loss:     {te_loss:.5f} | Test Accuracy:     {te_acc:.5f}% | Test F1: {epoch_f1:.2f}%")
            print(f"Mean Grad:     {mean_grad:.5f} | Mean Lambda (λ):   {mean_lam:.4f}")
            print("=======================================================================")
        else:
            te_loss, te_acc, epoch_f1 = history["test_loss"][-1], history["test_acc"][-1], history["test_f1"][-1]

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_loss"].append(te_loss)
        history["test_acc"].append(te_acc)
        history["test_f1"].append(epoch_f1)
        history["grad_norms"].append(grads)
        history["lambdas"].append(lams)

    cnn_results[name] = history

    checkpoint_payload = {
        "architecture": "CNN",
        "activation_name": name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": epochs
    }
    save_path = os.path.join(checkpoint_dir, f"cnn_baseline_{name}.pth")
    torch.save(checkpoint_payload, save_path)
    print(f"[✓] Saved CNN checkpoint: {save_path}\n")

print("\n[✓] CNN Baseline Training Completed!")

                   PHASE 1: TRAINING CNN BASELINES                    
               TRAINING WITH ACTIVATION: RELU               


Training RELU:   0%|          | 0/60 [00:00<?, ?epoch/s]

# **CNN Analysis — Accuracy and Loss Curves**

In [ ]:
plt.figure(figsize=(15, 5))

# Subplot 1: Test Accuracy
plt.subplot(1, 2, 1)
for name, hist in cnn_results.items():
    plt.plot(range(1, epochs + 1), hist["test_acc"], label=name.upper(), linewidth=2)
plt.title("CNN: CIFAR-10 Test Accuracy Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Subplot 2: Test Loss
plt.subplot(1, 2, 2)
for name, hist in cnn_results.items():
    plt.plot(range(1, epochs + 1), hist["test_loss"], label=name.upper(), linewidth=2)
plt.title("CNN: CIFAR-10 Test Loss Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cnn_accuracy_loss_curves.png"), dpi=300, bbox_inches="tight")
plt.show()

# **CNN Analysis — Gradient Norms & Lambda Trajectory**

In [ ]:
plt.figure(figsize=(15, 5))

# Subplot 1: Layer-Wise Gradient Norms
plt.subplot(1, 2, 1)
for name in baselines:
    grad_history = cnn_results[name]["grad_norms"]
    if grad_history and len(grad_history[-1]) > 0:
        final_norms = list(grad_history[-1].values())
        mean_grad = sum(final_norms) / len(final_norms)
        plt.bar(name.upper(), mean_grad, alpha=0.85)
plt.title("CNN: Mean Layer-Wise Gradient Norm at Final Epoch", fontsize=12, fontweight="bold")
plt.ylabel("Gradient Norm Magnitude")
plt.grid(axis="y", linestyle="--", alpha=0.6)

# Subplot 2: Sharpness Parameter Trajectory
plt.subplot(1, 2, 2)
for name in ["lambdagelu", "mpgelu"]:
    if name in cnn_results:
        lambdas = cnn_results[name]["lambdas"]
        if lambdas and len(lambdas[0]) > 0:
            avg_lambdas = [sum(l) / len(l) for l in lambdas]
            plt.plot(range(1, epochs + 1), avg_lambdas, label=f"{name.upper()}", linewidth=2, marker="o")

plt.axhline(y=1.0, color="r", linestyle=":", label="Lower Bound Constraint ($\lambda = 1.0$)")
plt.title("CNN: Sharpness Parameter ($\lambda$) Drift", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Lambda Parameter ($\lambda$)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cnn_gradient_and_lambda_metrics.png"), dpi=300, bbox_inches="tight")
plt.show()

# **CNN Analysis — Performance Summary Table**

In [ ]:
cnn_summary_metrics = []
for name in baselines:
    file_path = os.path.join(checkpoint_dir, f"cnn_baseline_{name}.pth")
    if os.path.exists(file_path):
        chkpt = torch.load(file_path, map_location="cpu")
        hist = chkpt["history"]

        cnn_summary_metrics.append({
            "Activation": name.upper(),
            "Final Test Acc (%)": round(hist["test_acc"][-1], 2),
            "Best Test Acc (%)": round(max(hist["test_acc"]), 2),
            "Final Test Loss": round(hist["test_loss"][-1], 4),
            "Min Test Loss": round(min(hist["test_loss"]), 4),
            "Final F1-Score (%)": round(hist["test_f1"][-1], 2),
            "Best F1-Score (%)": round(max(hist["test_f1"]), 2)
        })

cnn_summary_df = pd.DataFrame(cnn_summary_metrics)
print("=================== CNN PERFORMANCE SUMMARY ===================")
cnn_summary_df

# **PHASE 2(MLP):**

# **Execution Loop — MLP Baselines Training & Auto-Saving**

In [ ]:
print("=======================================================================")
print("                   PHASE 2: TRAINING MLP BASELINES                    ")
print("=======================================================================")

mlp_hidden_layers = [1024, 512, 256, 128]
mlp_results = {}

for name in baselines:
    print("=======================================================================")
    print(f"               TRAINING WITH ACTIVATION: {name.upper()}               ")
    print("=======================================================================")

    act_class = get_activation(name)

    model = MyMLP(input_dim=3072, output_dim=10, hidden_dims=mlp_hidden_layers, activation=act_class).to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    trainer = Trainer(
        model=model,
        loss_fn=loss_fn,
        optimizer=optimizer,
        calculate_accuracy=accuracy_fn,
        device=device,
        loss_steps=1
    )

    history = {
        "train_loss": [], "train_acc": [],
        "test_loss": [], "test_acc": [], "test_f1": [],
        "grad_norms": [], "lambdas": []
    }

    epoch_bar = tqdm(range(1, epochs + 1), desc=f"Training {name.upper()}", unit="epoch")

    for epoch in epoch_bar:
        tr_loss, tr_acc, grads, lams = trainer.train(train_loader, epoch=epoch)

        if epoch % 5 == 0 or epoch == epochs:
            te_loss, te_acc = trainer.test(test_loader, epoch=epoch)

            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for X, y in test_loader:
                    X, y = X.to(device), y.to(device)
                    preds = model(X).argmax(dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(y.cpu().numpy())

            epoch_f1 = f1_score(all_targets, all_preds, average="macro") * 100.0

            mean_grad = sum(grads.values()) / len(grads) if grads else 0.0
            mean_lam = sum(lams) / len(lams) if lams else 0.0

            print(f"Epoch: {epoch}")
            print("-----------------------------------------------------------------------")
            print(f"Training Loss: {tr_loss:.5f} | Training Accuracy: {tr_acc:.5f}%")
            print(f"Test Loss:     {te_loss:.5f} | Test Accuracy:     {te_acc:.5f}% | Test F1: {epoch_f1:.2f}%")
            print(f"Mean Grad:     {mean_grad:.5f} | Mean Lambda (λ):   {mean_lam:.4f}")
            print("=======================================================================")
        else:
            te_loss, te_acc, epoch_f1 = history["test_loss"][-1], history["test_acc"][-1], history["test_f1"][-1]

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_loss"].append(te_loss)
        history["test_acc"].append(te_acc)
        history["test_f1"].append(epoch_f1)
        history["grad_norms"].append(grads)
        history["lambdas"].append(lams)

    mlp_results[name] = history

    checkpoint_payload = {
        "architecture": "MLP",
        "activation_name": name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "epochs": epochs
    }
    save_path = os.path.join(checkpoint_dir, f"mlp_baseline_{name}.pth")
    torch.save(checkpoint_payload, save_path)
    print(f"[✓] Saved MLP checkpoint: {save_path}\n")

print("\n[✓] MLP Baseline Training Completed!")

# **MLP Analysis — Accuracy and Loss Curves**

In [ ]:
plt.figure(figsize=(15, 5))

# Subplot 1: Test Accuracy
plt.subplot(1, 2, 1)
for name, hist in mlp_results.items():
    plt.plot(range(1, epochs + 1), hist["test_acc"], label=name.upper(), linewidth=2, linestyle="--")
plt.title("MLP: CIFAR-10 Test Accuracy Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Subplot 2: Test Loss
plt.subplot(1, 2, 2)
for name, hist in mlp_results.items():
    plt.plot(range(1, epochs + 1), hist["test_loss"], label=name.upper(), linewidth=2, linestyle="--")
plt.title("MLP: CIFAR-10 Test Loss Comparison", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "mlp_accuracy_loss_curves.png"), dpi=300, bbox_inches="tight")
plt.show()

# **MLP Analysis — Gradient Norms & Lambda Trajectory**

In [ ]:
plt.figure(figsize=(15, 5))

# Subplot 1: Layer-Wise Gradient Norms
plt.subplot(1, 2, 1)
for name in baselines:
    grad_history = mlp_results[name]["grad_norms"]
    if grad_history and len(grad_history[-1]) > 0:
        final_norms = list(grad_history[-1].values())
        mean_grad = sum(final_norms) / len(final_norms)
        plt.bar(name.upper(), mean_grad, alpha=0.85)
plt.title("MLP: Mean Layer-Wise Gradient Norm at Final Epoch", fontsize=12, fontweight="bold")
plt.ylabel("Gradient Norm Magnitude")
plt.grid(axis="y", linestyle="--", alpha=0.6)

# Subplot 2: Sharpness Parameter Trajectory
plt.subplot(1, 2, 2)
for name in ["lambdagelu", "mpgelu"]:
    if name in mlp_results:
        lambdas = mlp_results[name]["lambdas"]
        if lambdas and len(lambdas[0]) > 0:
            avg_lambdas = [sum(l) / len(l) for l in lambdas]
            plt.plot(range(1, epochs + 1), avg_lambdas, label=f"{name.upper()}", linewidth=2, marker="s", linestyle="--")

plt.axhline(y=1.0, color="r", linestyle=":", label="Lower Bound Constraint ($\lambda = 1.0$)")
plt.title("MLP: Sharpness Parameter ($\lambda$) Drift", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Lambda Parameter ($\lambda$)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "mlp_gradient_and_lambda_metrics.png"), dpi=300, bbox_inches="tight")
plt.show()

# **MLP Analysis — Performance Summary Table**

In [ ]:
mlp_summary_metrics = []
for name in baselines:
    file_path = os.path.join(checkpoint_dir, f"mlp_baseline_{name}.pth")
    if os.path.exists(file_path):
        chkpt = torch.load(file_path, map_location="cpu")
        hist = chkpt["history"]

        mlp_summary_metrics.append({
            "Activation": name.upper(),
            "Final Test Acc (%)": round(hist["test_acc"][-1], 2),
            "Best Test Acc (%)": round(max(hist["test_acc"]), 2),
            "Final Test Loss": round(hist["test_loss"][-1], 4),
            "Min Test Loss": round(min(hist["test_loss"]), 4),
            "Final F1-Score (%)": round(hist["test_f1"][-1], 2),
            "Best F1-Score (%)": round(max(hist["test_f1"]), 2)
        })

mlp_summary_df = pd.DataFrame(mlp_summary_metrics)
print("=================== MLP PERFORMANCE SUMMARY ===================")
mlp_summary_df

# **PHASE 3(CROSS-ARCHITECTURE COMPARISON & SYNTHESIS):**

# **Cross-Architecture Plot — Side-by-Side Accuracy & Loss**

In [ ]:
plt.figure(figsize=(16, 10))

# Subplot 1: CNN Test Accuracy
plt.subplot(2, 2, 1)
for name, hist in cnn_results.items():
    plt.plot(range(1, epochs + 1), hist["test_acc"], label=name.upper(), linewidth=2)
plt.title("CNN: Test Accuracy", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Subplot 2: MLP Test Accuracy
plt.subplot(2, 2, 2)
for name, hist in mlp_results.items():
    plt.plot(range(1, epochs + 1), hist["test_acc"], label=name.upper(), linewidth=2, linestyle="--")
plt.title("MLP: Test Accuracy", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Subplot 3: CNN Test Loss
plt.subplot(2, 2, 3)
for name, hist in cnn_results.items():
    plt.plot(range(1, epochs + 1), hist["test_loss"], label=name.upper(), linewidth=2)
plt.title("CNN: Test Loss", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

# Subplot 4: MLP Test Loss
plt.subplot(2, 2, 4)
for name, hist in mlp_results.items():
    plt.plot(range(1, epochs + 1), hist["test_loss"], label=name.upper(), linewidth=2, linestyle="--")
plt.title("MLP: Test Loss", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cross_architecture_side_by_side.png"), dpi=300, bbox_inches="tight")
plt.show()

# **Cross-Architecture Plot — Lambda Drift Overlay (CNN vs. MLP)**

In [ ]:
plt.figure(figsize=(12, 5))

# CNN Sharpness Drift
for name in ["lambdagelu", "mpgelu"]:
    if name in cnn_results:
        lambdas = cnn_results[name]["lambdas"]
        if lambdas and len(lambdas[0]) > 0:
            avg_lambdas = [sum(l) / len(l) for l in lambdas]
            plt.plot(range(1, epochs + 1), avg_lambdas, label=f"CNN {name.upper()}", linewidth=2, marker="o")

# MLP Sharpness Drift
for name in ["lambdagelu", "mpgelu"]:
    if name in mlp_results:
        lambdas = mlp_results[name]["lambdas"]
        if lambdas and len(lambdas[0]) > 0:
            avg_lambdas = [sum(l) / len(l) for l in lambdas]
            plt.plot(range(1, epochs + 1), avg_lambdas, label=f"MLP {name.upper()}", linewidth=2, marker="s", linestyle="--")

plt.axhline(y=1.0, color="r", linestyle=":", label="Lower Bound Constraint ($\lambda = 1.0$)")
plt.title("Cross-Architecture Sharpness Drift ($\lambda$): CNN vs. MLP", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Lambda Parameter ($\lambda$)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(checkpoint_dir, "cross_architecture_lambda_drift.png"), dpi=300, bbox_inches="tight")
plt.show()

# **Unified Cross-Architecture Quantitative Summary Table**

In [ ]:
combined_summary_metrics = []

# Load CNN records from disk
for name in baselines:
    file_path = os.path.join(checkpoint_dir, f"cnn_baseline_{name}.pth")
    if os.path.exists(file_path):
        chkpt = torch.load(file_path, map_location="cpu")
        hist = chkpt["history"]

        combined_summary_metrics.append({
            "Architecture": "CNN",
            "Activation": name.upper(),
            "Final Test Acc (%)": round(hist["test_acc"][-1], 2),
            "Best Test Acc (%)": round(max(hist["test_acc"]), 2),
            "Final Test Loss": round(hist["test_loss"][-1], 4),
            "Min Test Loss": round(min(hist["test_loss"]), 4),
            "Final F1-Score (%)": round(hist["test_f1"][-1], 2),
            "Best F1-Score (%)": round(max(hist["test_f1"]), 2)
        })

# Load MLP records from disk
for name in baselines:
    file_path = os.path.join(checkpoint_dir, f"mlp_baseline_{name}.pth")
    if os.path.exists(file_path):
        chkpt = torch.load(file_path, map_location="cpu")
        hist = chkpt["history"]

        combined_summary_metrics.append({
            "Architecture": "MLP",
            "Activation": name.upper(),
            "Final Test Acc (%)": round(hist["test_acc"][-1], 2),
            "Best Test Acc (%)": round(max(hist["test_acc"]), 2),
            "Final Test Loss": round(hist["test_loss"][-1], 4),
            "Min Test Loss": round(min(hist["test_loss"]), 4),
            "Final F1-Score (%)": round(hist["test_f1"][-1], 2),
            "Best F1-Score (%)": round(max(hist["test_f1"]), 2)
        })

combined_summary_df = pd.DataFrame(combined_summary_metrics)
print("=================== UNIFIED CROSS-ARCHITECTURE BENCHMARK ===================")
combined_summary_df